In [18]:
using Revise
using Flux
using Flux: glorot_uniform
using JLD2
using CUDA
using WaterLily

includet("../custom.jl")

In [ ]:
@load "/home/matth/Thesis/data/RHS_shedding_data_arr.jld2" RHS_data
downsample_RHS_data!(RHS_data; tmin=50, tmax=75, n_samples=50)

snapshot = cat(RHS_data["RHS"][1:10]...; dims=4)  # shape: (386, 130, 2, 10)
println("Input snapshot size: ", size(snapshot))

H, W, C, T = size(snapshot)

Downsampled to 50 time steps.
Input data size: (386, 130, 2)
Input snapshot size: (386, 130, 2, 10)


(386, 130, 2, 10)

In [ ]:
# encoder parameters
n_latent = 128  # Latent space dimension
C_next = 4
kernel_size = (3, 3)
padding=1

# Convolutional encoder
kernel1 = (3, 3)
kernel2 = (5, 5)
kernel3 = (7, 7)
kernel4 = (9, 9)

pooling_encoder = Chain(
    Conv(kernel1, C => C_next, relu; pad=(1, 1)),       
    MaxPool((2, 2)),                                     
    Conv(kernel2, C_next => 2C_next, relu; pad=(1, 1)), 
    MaxPool((2, 2)),
    Conv(kernel3, 2C_next => 4C_next, relu; pad=(1, 1)),
    MaxPool((2, 2)),
    Conv(kernel4, 4C_next => 8C_next, relu; pad=(1, 1)),
    MaxPool((2, 2)),
    x -> reshape(x, :, size(x, 4)),  # Flatten
)



Chain(
  Conv((3, 3), 2 => 4, relu, pad=1),    # 76 parameters
  MaxPool((2, 2)),
  Conv((5, 5), 4 => 8, relu, pad=1),    # 808 parameters
  MaxPool((2, 2)),
  Conv((7, 7), 8 => 16, relu, pad=1),   # 6_288 parameters
  MaxPool((2, 2)),
  Conv((9, 9), 16 => 32, relu, pad=1),  # 41_504 parameters
  MaxPool((2, 2)),
  var"#33#34"(),
)                   # Total: 8 arrays, 48_676 parameters, 191.078 KiB.

In [9]:
mid = Chain(
    x -> Dense(size(x, 1), n_latent)(x)  # Parametric Dense layer
)

Chain(#35)

In [10]:
decoder = Chain(
    # z : (n_latent, T) -> (24*8*8C_next, T)
    x -> Dense(n_latent, 24 * 8 * (8C_next))(x),
    # -> (24, 8, 8C_next, T)
    x -> reshape(x, 24, 8, 8C_next, size(x, 2)),

    # Upsample back to original spatial size via stride-2 deconvs
    # 24×8×(8C_next) -> 48×16×(4C_next)
    ConvTranspose((2, 2), 8C_next => 4C_next, relu; stride=(2, 2), pad=(0, 0)),
    # 48×16×(4C_next) -> 96×32×(2C_next)
    ConvTranspose((2, 2), 4C_next => 2C_next, relu; stride=(2, 2), pad=(0, 0)),
    # 96×32×(2C_next) -> 192×64×(C_next)
    ConvTranspose((2, 2), 2C_next => C_next,   relu; stride=(2, 2), pad=(0, 0)),
    # 192×64×(C_next) -> 384×128×C  (C is original input channels, here 2)
    ConvTranspose((2, 2), C_next   => C,       relu; stride=(2, 2), pad=(0, 0)),

    # Now 384×128×C; add +2 in both H and W to match 386×130
    # Keep identity activation for reconstruction
    ConvTranspose((3, 3), C => C, identity; stride=(1, 1), pad=(0, 0))
)

Chain(
  var"#37#39"(),
  var"#38#40"(),
  ConvTranspose((2, 2), 32 => 16, relu, stride=2),  # 2_064 parameters
  ConvTranspose((2, 2), 16 => 8, relu, stride=2),  # 520 parameters
  ConvTranspose((2, 2), 8 => 4, relu, stride=2),  # 132 parameters
  ConvTranspose((2, 2), 4 => 2, relu, stride=2),  # 34 parameters
  ConvTranspose((3, 3), 2 => 2),        # 38 parameters
)                   # Total: 10 arrays, 2_788 parameters, 11.828 KiB.

In [ ]:
Autoencoder = Chain(pooling_encoder, mid, decoder)

Chain(
  Chain(
    Conv((3, 3), 2 => 4, relu, pad=1),  # 76 parameters
    MaxPool((2, 2)),
    Conv((5, 5), 4 => 8, relu, pad=1),  # 808 parameters
    MaxPool((2, 2)),
    Conv((7, 7), 8 => 16, relu, pad=1),  # 6_288 parameters
    MaxPool((2, 2)),
    Conv((9, 9), 16 => 32, relu, pad=1),  # 41_504 parameters
    MaxPool((2, 2)),
    var"#33#34"(),
  ),
  Chain(#35),
  Chain(
    var"#37#39"(),
    var"#38#40"(),
    ConvTranspose((2, 2), 32 => 16, relu, stride=2),  # 2_064 parameters
    ConvTranspose((2, 2), 16 => 8, relu, stride=2),  # 520 parameters
    ConvTranspose((2, 2), 8 => 4, relu, stride=2),  # 132 parameters
    ConvTranspose((2, 2), 4 => 2, relu, stride=2),  # 34 parameters
    ConvTranspose((3, 3), 2 => 2),      # 38 parameters
  ),
)                   # Total: 18 arrays, 51_464 parameters, 202.906 KiB.

In [ ]:
output = pooling_encoder(snapshot)
println("Output size: ", size(output))

CR = (386 * 130 * 2) / size(output, 1) 
println("Compression ratio: ", CR)


ae_output = Autoencoder(snapshot)
println("Autoencoder output size: ", size(ae_output))

In [ ]:
# --- loss ---

function recon_loss(ae, x)
    ŷ = ae(x)
    return mse(ŷ, x)
end

function div_penalty(x; λₚ=1)
    return WaterLily.div(x) * λₚ * 0
end

function total_loss(ae, x; λₚ=1)
    𝓛rec = recon_loss(ae, x)
    𝓛div = div_penalty(ae(x); λₚ=λₚ)
    return 𝓛rec + 𝓛div, (𝓛rec, 𝓛div)
end

div_penalty (generic function with 1 method)

In [17]:
# --- GPU if available ---
println("CUDA functional: ", CUDA.functional())
dev(x) = CUDA.functional() ? cu(x) : x
ae = dev(ae)
snapshot = dev(snapshot)


CUDA functional: true


386×130×2×10 CuArray{Float32, 4, CUDA.DeviceMemory}:
[:, :, 1, 1] =
 0.0   0.0           0.0           0.0          …   0.0           0.0
 0.0  -0.14816      -0.148149     -0.148126        -0.098492     -0.000396848
 0.0  -1.90405f-5   -1.88453f-5   -1.87152f-5       1.93011f-5   -0.00235486
 0.0  -2.78288f-5   -2.78745f-5   -2.82747f-5       2.88675f-5   -0.00188947
 0.0  -3.94844f-5   -4.01367f-5   -3.99826f-5       4.05679f-5   -0.00189769
 0.0  -5.08537f-5   -5.02581f-5   -5.04188f-5   …   5.25091f-5   -0.00190866
 0.0  -6.19631f-5   -6.19837f-5   -6.23446f-5       6.3471f-5    -0.00192308
 0.0  -7.3084f-5    -7.30623f-5   -7.27673f-5       7.55309f-5   -0.00193942
 0.0  -8.35002f-5   -8.4101f-5    -8.47552f-5       8.72391f-5   -0.00195885
 0.0  -9.41928f-5   -9.44508f-5   -9.44538f-5       9.82956f-5   -0.00198054
 ⋮                                              ⋱                
 0.0  -0.00202794   -0.00203775   -0.00205321       0.00177889    0.00475383
 0.0  -0.00177135   -0.00

In [ ]:
batchsize = min(4, length(trn_idx))
epochs    = 50
opt       = Adam(1e-3)
clip_val  = 1.0   # gradient clipping
best_val  = Inf